In [8]:
import os
import sys
import json

from pathlib import Path


path_proj = Path.cwd().parent
path_data = os.path.join(path_proj, "data")


sys.path.append(os.path.join(path_proj, "utils"))


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# from generate_llm import generate_gpt

# generate_gpt("What is the capital of France?")

In [37]:
from fed_extract_links import (extract_index_fomc_materials,
                               get_statements_minutes_links)

# extract_index_fomc_materials(path_data)


In [38]:
get_statements_minutes_links()

['https://www.federalreserve.gov/newsevents/pressreleases/monetary20260128a.htm',
 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20260128a1.htm',
 'https://www.federalreserve.gov/monetarypolicy/fomcpressconf20260128.htm',
 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20260128b.htm',
 'https://www.federalreserve.gov/monetarypolicy/fomcminutes20260128.htm',
 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20260318a.htm',
 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20260318a1.htm',
 'https://www.federalreserve.gov/monetarypolicy/fomcprojtabl20260318.htm',
 'https://www.federalreserve.gov/monetarypolicy/fomcminutes20260318.htm',
 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a.htm',
 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20250129a1.htm',
 'https://www.federalreserve.gov/monetarypolicy/fomcminutes20250129.htm',
 'https://www.federalreserve.gov/newsevents/pressr

In [41]:
import re
from urllib.parse import urljoin

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


SPEECH_SELECTOR = 'a[href*="/newsevents/speech/"]'
TESTIMONY_SELECTOR = 'a[href*="/newsevents/testimony/"]'
NEXT_BUTTON_XPATH = (
    "//a[normalize-space()='Next' or contains(@aria-label, 'Next') or contains(@title, 'Next')]"
    " | "
    "//button[normalize-space()='Next' or contains(@aria-label, 'Next') or contains(@title, 'Next')]"
)

ENTRY_HREF_PATTERN = re.compile(
    r"/newsevents/(?P<kind>speech|testimony)/[a-z]+(?P<date>\d{8})(?P<suffix>[a-z])\.htm$",
    re.IGNORECASE,
)


def _entry_anchors(driver):
    anchors = driver.find_elements(
        By.CSS_SELECTOR,
        f"{SPEECH_SELECTOR}, {TESTIMONY_SELECTOR}",
    )
    valid_anchors = []

    for anchor in anchors:
        href = anchor.get_attribute("href")
        title = anchor.text.strip()

        if not href or not title:
            continue

        if not ENTRY_HREF_PATTERN.search(href):
            continue

        valid_anchors.append(anchor)

    return valid_anchors


def _wait_for_results_page(wait, min_results=10):
    wait.until(lambda d: len(_entry_anchors(d)) >= min_results)


def _get_page_marker(driver):
    anchors = _entry_anchors(driver)
    if anchors:
        return anchors[0].get_attribute("href")
    return None


def _extract_entries_from_current_page(driver, seen, page_number, base_url):
    page_entries = []

    for anchor in _entry_anchors(driver):
        href = anchor.get_attribute("href")
        title = anchor.text.strip()

        match = ENTRY_HREF_PATTERN.search(href)
        if not match or href in seen:
            continue

        seen.add(href)

        page_entries.append(
            {
                "title": title,
                "url": urljoin(base_url, href),
                "date": match.group("date"),
                "type": match.group("kind"),
                "page": page_number,
            }
        )

    return page_entries


def _go_to_next_page(driver, wait, previous_marker):
    next_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, NEXT_BUTTON_XPATH))
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        next_button,
    )
    driver.execute_script("arguments[0].click();", next_button)

    wait.until(
        lambda d: (_get_page_marker(d) is not None)
        and (_get_page_marker(d) != previous_marker)
    )

    _wait_for_results_page(wait)


def extract_speeches_testimony_links(
    url="https://www.federalreserve.gov/newsevents/speeches-testimony.htm",
    next_pages=4,
):
    driver = webdriver.Edge()
    wait = WebDriverWait(driver, 35)

    try:
        driver.get(url)

        last_update = wait.until(
            EC.presence_of_element_located((By.CLASS_NAME, "lastUpdate"))
        ).text
        last_update = last_update.split(":", 1)[-1].strip()

        _wait_for_results_page(wait)

        seen = set()
        all_entries = []
        total_pages = next_pages + 1

        for page_number in range(1, total_pages + 1):
            all_entries.extend(
                _extract_entries_from_current_page(driver, seen, page_number, url)
            )

            if page_number == total_pages:
                break

            previous_marker = _get_page_marker(driver)

            try:
                _go_to_next_page(driver, wait, previous_marker)
            except TimeoutException:
                break

        return {
            "last_update": last_update,
            "count": len(all_entries),
            "pages_scraped": page_number,
            "links": all_entries,
        }

    finally:
        driver.quit()

In [43]:
result = extract_speeches_testimony_links(next_pages=4)

result["pages_scraped"], result["count"]
result["links"]

[{'title': 'Rural Communities: Worth the Investment',
  'url': 'https://www.federalreserve.gov/newsevents/speech/barr20260414a.htm',
  'date': '20260414',
  'type': 'speech',
  'page': 1},
 {'title': 'Economic Outlook and the Labor Market',
  'url': 'https://www.federalreserve.gov/newsevents/speech/jefferson20260407a.htm',
  'date': '20260407',
  'type': 'speech',
  'page': 1},
 {'title': 'Supporting Small Businesses',
  'url': 'https://www.federalreserve.gov/newsevents/speech/bowman20260331a.htm',
  'date': '20260331',
  'type': 'speech',
  'page': 1},
 {'title': 'Brief Remarks on Stablecoins',
  'url': 'https://www.federalreserve.gov/newsevents/speech/barr20260331a.htm',
  'date': '20260331',
  'type': 'speech',
  'page': 1},
 {'title': 'Brief Remarks on the Economic Outlook and Monetary Policy',
  'url': 'https://www.federalreserve.gov/newsevents/speech/barr20260326a.htm',
  'date': '20260326',
  'type': 'speech',
  'page': 1},
 {'title': 'Economic Outlook and Energy Effects',
  'ur